In [0]:
from bs4 import BeautifulSoup
from functools import reduce
import pyspark.sql.functions as f
import pyspark.pipelines as dp
import requests
import pyspark

In [0]:
venue_initials = {
    "T4B": "4Bs Tavern",
    "8BT": "8-Bit Arcade Bar",
    "AAB": "Add-a-Ball",
    "ADM": "Admiral Pub",
    "ANC": "Another Castle",
    "BSS": "Ballard Smoke Shop",
    "COP": "Corner Pocket Billiards and Lounge",
    "GRY": "Gary's Place",
    "GPA": "Georgetown Pizza and Arcade",
    "HND": "Hounds Tooth",
    "IBX": "Ice Box",
    "JUP": "Jupiter",
    "KRA": "Kraken",
    "OLF": "Olaf's",
    "RSW": "Rickshaw",
    "STN": "Seattle Tavern and Pool Hall",
    "SHR": "Shorty's",
    "TLT": "Tilted Table",
    "TWP": "Time Warp",
    "TDN": "Touchdown",
    "WAT": "Waterland",
    "ZTE": "Zoo Tavern",
}

In [0]:
venue_df = spark.createDataFrame(
    [(k, v) for k, v in venue_initials.items()],
    schema="venue_init STRING, venue_name STRING"
)

display(venue_df)

In [0]:
def html_extract(url: str):

    html_src = requests.get(url).content
    soup = BeautifulSoup(html_src, "html.parser")
    tables = soup.find_all('table')

    # Extract data from HTML
    for table in tables:
        rows = table.find_all('tr')

        # First row contains the headers
        header_cells = rows[0].find_all(['th', 'td'])
        headers = [cell.text.strip() or f"column_{i}" for i, cell in enumerate(header_cells)]

        # Remaining rows are data
        table_data = []
        for row in rows[1:]:
            cells = row.find_all('td')
            row_dict = {headers[i]: cell.text.strip() for i, cell in enumerate(cells) if i < len(headers)}
            table_data.append(row_dict)

    return table_data

In [0]:
data = html_extract("https://mondaynightpinball.com/machines")

df = spark.createDataFrame(data)

In [0]:
display(df)

In [0]:
from pyspark.sql import Row

# Create a single-row DataFrame from the column names, then union with the original
header_row = spark.createDataFrame([Row(*df.columns)], schema=df.schema)
df = header_row.unionByName(df)
# Rename columns
df = df.withColumnRenamed("AC/DC", "machine_name").withColumnRenamed("ACDC", "machine_abbrv")


display(df)